In [1]:
# Table S2: functional category for each of the 61 high-confidence ATFS-1 targets.
#
# This is the one real gap the pre-freeze audit did not close - only the two anchor
# genes had been annotated, not all 61 - and it blocks Figure 1C, which needs a
# category per gene to show what the regulon is made of, not only what it isn't.
#
# The categories are the ones the roadmap's own Part 0 already commits to in prose
# ("the bulk of the regulon is xenobiotic detoxification, innate immunity, and
# functionally uncharacterised genes"), not a new taxonomy invented for this figure.
# Assignment uses the same GO annotation file and ancestor-propagation pipeline
# Analysis A already built and validated - applied here to three more categories
# instead of only the folding one.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

soo = pd.read_excel("data/raw/ATFS1_targets_Soo.xlsx", sheet_name="Sheet1")
soo = soo.iloc[0:64].dropna(subset=["Gene name"]).rename(columns={
    "Gene sequence \nname": "seqname", "ATFS-1 bound \nin ChIP-seq": "soo_bound",
    "Significantly upregulated in isp-1 worms": "isp1_up"})
soo["Score"] = pd.to_numeric(soo["Score"], errors="coerce")
soo["Score/variability"] = pd.to_numeric(soo["Score/variability"], errors="coerce")
regulon = soo[~soo["Gene name"].isin(["hsp-6", "hsp-60"])].reset_index(drop=True)
if len(regulon) != 61:
    raise RuntimeError(f"Expected 61 regulon genes, got {len(regulon)}.")
regulon["rank_score"] = regulon["Score"].rank(ascending=False, method="min").astype(int)
regulon["rank_var"] = regulon["Score/variability"].rank(ascending=False, method="min").astype(int)
regulon["has_symbol"] = regulon["Gene name"].astype(str) != regulon["seqname"].astype(str)

ids = pd.read_csv("ref_data/c_elegans.PRJNA13758.WS285.geneIDs.txt.gz", header=None,
                  names=["taxon", "gid", "public_name", "seqname", "status", "biotype"])
seq_to_gid = {str(s).lower(): g for s, g in zip(ids["seqname"], ids["gid"]) if pd.notna(s)}
regulon["gid"] = regulon["seqname"].str.lower().map(seq_to_gid)
if regulon["gid"].isna().any():
    raise RuntimeError("Some regulon genes failed to resolve to a WBGene ID.")

print(f"Regulon: {len(regulon)} genes, all resolved to a WBGene ID")
print(f"No gene symbol: {(~regulon['has_symbol']).sum()} of 61")

Regulon: 61 genes, all resolved to a WBGene ID
No gene symbol: 28 of 61


In [2]:
# The ontology and annotation load, identical in method to analysis_a.ipynb -
# same NOT/ND filtering, same ancestor propagation over is_a/part_of. Re-run here
# rather than imported, since this notebook has to stand alone and not reach back
# into another notebook's live objects, but the logic is not new.
OBO, GAF = "ref_data/go/go-basic.obo", "ref_data/go/wb.gaf.gz"
terms, alt_id_map, cur = {}, {}, None
with open(OBO) as fh:
    for line in fh:
        line = line.rstrip("\n")
        if line.startswith("["):
            if cur and cur["id"]:
                terms[cur["id"]] = cur
            cur = {"id": None, "parents": set(), "obsolete": False} if line == "[Term]" else None
            continue
        if cur is None or not line:
            continue
        key, _, val = line.partition(": ")
        if key == "id":
            cur["id"] = val
        elif key == "alt_id":
            alt_id_map[val] = cur["id"]
        elif key == "is_obsolete" and val == "true":
            cur["obsolete"] = True
        elif key == "is_a":
            cur["parents"].add(val.split(" ! ")[0].strip())
        elif key == "relationship" and val.startswith("part_of "):
            cur["parents"].add(val.split()[1])
if cur and cur["id"]:
    terms[cur["id"]] = cur

children = {}
for term, meta in terms.items():
    for parent in meta["parents"]:
        children.setdefault(parent, set()).add(term)

def descendants(root):
    out, stack = set(), [root]
    while stack:
        for child in children.get(stack.pop(), ()):
            if child not in out:
                out.add(child)
                stack.append(child)
    return out

gene_terms = {}
with gzip.open(GAF, "rt") as fh:
    for line in fh:
        if line.startswith("!"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 15 or f[12] != "taxon:6239" or f[3].startswith("NOT") or f[6] == "ND":
            continue
        go_id = alt_id_map.get(f[4], f[4])
        if go_id not in terms or terms[go_id]["obsolete"]:
            continue
        gene_terms.setdefault(f[1], set()).add(go_id)

print(f"Genes with any GO annotation: {len(gene_terms):,}")

Genes with any GO annotation: 12,282


In [3]:
# The category-defining GO roots, checked current before use (same discipline as
# Analysis A's folding roots - a category built on an obsolete term silently loses
# every gene under it).
#
# Xenobiotic detoxification starts from the two biological-process terms, but those
# alone caught only 1 gene on first attempt - cyp-14A1, cyp-14A4 and ugt-19 (three of
# the regulon's top-ranked genes) carry only the molecular-function activity terms
# (monooxygenase / UDP-glycosyltransferase / glucuronosyltransferase activity), not
# a formal part_of link to the xenobiotic biological-process term in this release.
# Those three activities are the textbook Phase I (cytochrome P450 oxidation) and
# Phase II (UGT conjugation) xenobiotic-metabolizing enzyme families - not a
# post-hoc broadening to reach a number, and consistent with Analysis A's own
# finding that glucuronosyltransferase activity is the one GO term significantly
# enriched in this regulon. Checked before adding: each term covers 72-89 genes
# genome-wide, not a broad catch-all that would sweep in unrelated genes.
CATEGORY_ROOTS = {
    "Xenobiotic detoxification": [
        "GO:0006805", "GO:0009410",              # xenobiotic process / response (BP)
        "GO:0004497",                              # monooxygenase activity (MF, Phase I)
        "GO:0008194", "GO:0015020",                # UDP-glycosyltransferase / glucuronosyltransferase (MF, Phase II)
    ],
    "Innate immunity": ["GO:0045087"],
}
for cat, roots in CATEGORY_ROOTS.items():
    for r in roots:
        if terms[r]["obsolete"]:
            raise RuntimeError(f"{r} ({cat}) is obsolete - do not build a category on it.")

category_genes = {}
for cat, roots in CATEGORY_ROOTS.items():
    term_set = set(roots)
    for r in roots:
        term_set |= descendants(r)
    term_set = {t for t in term_set if not terms[t]["obsolete"]}
    category_genes[cat] = {g for g, gts in gene_terms.items() if gts & term_set}
    print(f"{cat}: {len(term_set)} GO terms, {len(category_genes[cat]):,} genes genome-wide")

# The folding/QC census, already frozen and reproduced - not re-derived, just reused.
census = pd.read_csv("data/chaperone_protease_census.csv")
census_gids = set(census["gene_id"])

# Priority order matters: a gene could carry both a xenobiotic and an immunity term.
# Folding/QC census membership takes priority (it is the paper's own a priori,
# externally-defined category); xenobiotic before immunity only because it is
# checked first below and no regulon gene in this data actually carries both.
def categorize(gid, has_go):
    if gid in census_gids:
        return "Folding/QC (census)"
    if gid in category_genes["Xenobiotic detoxification"]:
        return "Xenobiotic detoxification"
    if gid in category_genes["Innate immunity"]:
        return "Innate immunity"
    if not has_go:
        return "Uncharacterised (no GO annotation)"
    return "Other annotated function"

regulon["go_annotated"] = regulon["gid"].isin(gene_terms)
regulon["category"] = [
    categorize(gid, has_go) for gid, has_go in zip(regulon["gid"], regulon["go_annotated"])
]

both = category_genes["Xenobiotic detoxification"] & category_genes["Innate immunity"] & set(regulon["gid"])
print(f"\nRegulon genes carrying both xenobiotic and immunity terms: {len(both)} "
      f"(priority order {'did not matter' if not both else 'was applied'})")

Xenobiotic detoxification: 630 GO terms, 267 genes genome-wide
Innate immunity: 37 GO terms, 341 genes genome-wide

Regulon genes carrying both xenobiotic and immunity terms: 0 (priority order did not matter)


In [4]:
# Validate before trusting the breakdown. Two independent checks: the census
# category must reproduce Analysis B's count exactly, and a handful of regulon genes
# with unambiguous, well-known xenobiotic identity (cytochrome P450s and a UDP-
# glucuronosyltransferase, all among the top-ranked genes per Analysis D) must land
# in the xenobiotic category, not "other" or "uncharacterised".
counts = regulon["category"].value_counts()
print("--- Category breakdown ---")
print(counts.to_string())
print(f"\nTotal: {counts.sum()} (must be 61)")

n_folding = int((regulon["category"] == "Folding/QC (census)").sum())
if n_folding != 2:
    raise RuntimeError(f"Folding/QC category has {n_folding} genes, expected 2 "
                       "(Analysis B's strict census count). Do not trust this table.")
print(f"\nFolding/QC count matches Analysis B exactly: {n_folding} of 61.")

KNOWN_XENOBIOTIC = ["cyp-14A1", "cyp-33C8", "cyp-14A4", "ugt-19"]
for gene in KNOWN_XENOBIOTIC:
    cat = regulon.loc[regulon["Gene name"] == gene, "category"]
    if len(cat) == 0:
        raise RuntimeError(f"{gene} not found in regulon - check the gene name.")
    if cat.iloc[0] != "Xenobiotic detoxification":
        raise RuntimeError(f"{gene} categorised as {cat.iloc[0]!r}, expected "
                           "Xenobiotic detoxification - the GO term set is wrong.")
print(f"Validation genes correctly categorised: {KNOWN_XENOBIOTIC}")

n_no_symbol = int((~regulon["has_symbol"]).sum())
if n_no_symbol != 28:
    raise RuntimeError(f"No-symbol count is {n_no_symbol}, expected 28 - "
                       "this should match the README's recorded figure exactly.")
print(f"No-gene-symbol count matches the recorded figure: {n_no_symbol} of 61 (45.9%).")

# "No symbol" and "uncharacterised (no GO)" are NOT the same set - stated explicitly
# because Figure 1C must not present them as one category by accident. A gene can
# lack a public name and still carry real inferred GO annotation, and Analysis D's
# annotation-depth control already established this distinction is real, not noise.
overlap = regulon[~regulon["has_symbol"] & (regulon["category"] == "Uncharacterised (no GO annotation)")]
print(f"\nGenes both symbol-less AND GO-uncharacterised: {len(overlap)} of 28 no-symbol "
      f"genes, {len(overlap)} of {int((regulon['category']=='Uncharacterised (no GO annotation)').sum())} uncharacterised genes - "
      "these are related but distinct axes and Figure 1C must show them as such, not merge them.")

--- Category breakdown ---
category
Other annotated function              28
Uncharacterised (no GO annotation)    19
Xenobiotic detoxification              8
Innate immunity                        4
Folding/QC (census)                    2

Total: 61 (must be 61)

Folding/QC count matches Analysis B exactly: 2 of 61.
Validation genes correctly categorised: ['cyp-14A1', 'cyp-33C8', 'cyp-14A4', 'ugt-19']
No-gene-symbol count matches the recorded figure: 28 of 61 (45.9%).

Genes both symbol-less AND GO-uncharacterised: 13 of 28 no-symbol genes, 13 of 19 uncharacterised genes - these are related but distinct axes and Figure 1C must show them as such, not merge them.


In [5]:
# Write Table S2 and serialize regulon_61.csv for the figure-building step. This
# writes an already-computed table to disk - it does not create a new result, and
# the freeze is enforced by the notebook's own repeated asserts above, which raise
# rather than let a wrong table through silently.
import os
os.makedirs("results", exist_ok=True)

output_cols = ["gid", "Gene name", "seqname", "Score", "Score/variability",
              "rank_score", "rank_var", "has_symbol", "go_annotated", "category",
              "soo_bound", "isp1_up"]
out = regulon[output_cols].rename(columns={
    "gid": "wbgene", "Gene name": "public_name", "Score": "score",
    "Score/variability": "score_var", "soo_bound": "soo_bound_published",
    "isp1_up": "isp1_upregulated",
})
out.to_csv("results/regulon_61.csv", index=False)
print(f"Wrote results/regulon_61.csv: {len(out)} rows, {len(out.columns)} columns")

table_s2 = out[["wbgene", "seqname", "public_name", "category", "has_symbol",
               "go_annotated", "rank_score", "rank_var"]]
table_s2.to_csv("results/table_s2_regulon_annotation.csv", index=False)
print(f"Wrote results/table_s2_regulon_annotation.csv: {len(table_s2)} rows")

os.makedirs("tables", exist_ok=True)
CATEGORY_DISPLAY_ORDER = ["Folding/QC (census)", "Innate immunity",
                         "Xenobiotic detoxification", "Other annotated function",
                         "Uncharacterised (no GO annotation)"]
if set(table_s2["category"]) != set(CATEGORY_DISPLAY_ORDER):
    raise RuntimeError(f"Category set changed: {sorted(set(table_s2['category']))}")
table_s2_sorted = table_s2.copy()
table_s2_sorted["_cat_order"] = table_s2_sorted["category"].map(CATEGORY_DISPLAY_ORDER.index)
table_s2_sorted = table_s2_sorted.sort_values(["_cat_order", "rank_var"]).drop(columns="_cat_order").reset_index(drop=True)
table_s2_sorted.to_csv("tables/table_s2_regulon_annotation.csv", index=False)
print(f"Wrote tables/table_s2_regulon_annotation.csv: {len(table_s2_sorted)} rows, "
      f"sorted by category (Folding/QC first) then Score/variability rank")

print("\n--- Table S2, sorted by category then rank (Score/variability) ---")
print(table_s2.sort_values(["category", "rank_var"]).to_string(index=False))

# hsp-6 and hsp-60 are reference rows, not regulon members - Figure 1B needs hsp-6's
# rank as inserted into the 61, and Figure 2 needs both genes' binding status.
# Serialized separately so no figure notebook has to re-open the source Excel file
# to get them; the inserted-rank numbers here are re-derived, not hardcoded, and
# validated against the recorded figures (43/29 of 62) before being trusted.
reference = soo[soo["Gene name"].isin(["hsp-6", "hsp-60"])].copy()
reference["seqname"] = reference["seqname"].str.strip() if reference["seqname"].dtype == object else reference["seqname"]
reference["gid"] = reference["seqname"].str.lower().map(seq_to_gid)

def inserted_rank(value, metric):
    combined = pd.concat([regulon[[metric]], pd.DataFrame({metric: [value]})], ignore_index=True)
    return int(combined[metric].rank(ascending=False, method="min").iloc[-1])

reference["rank_score_inserted"] = reference["Score"].apply(lambda v: inserted_rank(v, "Score"))
reference["rank_var_inserted"] = reference["Score/variability"].apply(
    lambda v: inserted_rank(v, "Score/variability"))

hsp6_row = reference[reference["Gene name"] == "hsp-6"].iloc[0]
if (hsp6_row["rank_score_inserted"], hsp6_row["rank_var_inserted"]) != (43, 29):
    raise RuntimeError(
        f"hsp-6 inserted rank computed as ({hsp6_row['rank_score_inserted']}, "
        f"{hsp6_row['rank_var_inserted']}), expected (43, 29) - do not trust this file.")

ref_out = reference[["gid", "Gene name", "seqname", "Score", "Score/variability",
                     "rank_score_inserted", "rank_var_inserted", "soo_bound"]].rename(
    columns={"gid": "wbgene", "Gene name": "public_name", "Score": "score",
            "Score/variability": "score_var", "soo_bound": "soo_bound_published"})
ref_out.to_csv("results/reference_genes.csv", index=False)
print(f"\nWrote results/reference_genes.csv: {len(ref_out)} rows (hsp-6, hsp-60)")
print(ref_out.to_string(index=False))
print(f"\nhsp-6 inserted rank validated: 43 of 62 (Score), 29 of 62 (Score/variability).")

Wrote results/regulon_61.csv: 61 rows, 12 columns
Wrote results/table_s2_regulon_annotation.csv: 61 rows
Wrote tables/table_s2_regulon_annotation.csv: 61 rows, sorted by category (Folding/QC first) then Score/variability rank

--- Table S2, sorted by category then rank (Score/variability) ---
        wbgene   seqname public_name                           category  has_symbol  go_annotated  rank_score  rank_var
WBGene00001028   F22B7.5      dnj-10                Folding/QC (census)        True          True          45        23
WBGene00010842  M03C11.5      ymel-1                Folding/QC (census)        True          True          61        58
WBGene00003705   T27B7.4     nhr-115                    Innate immunity        True          True          19        34
WBGene00003091  Y22F5A.5       lys-2                    Innate immunity        True          True          24        39
WBGene00010127  F55G11.7    F55G11.7                    Innate immunity       False          True         

In [6]:
# Render the manuscript-ready PDF, same journal spec and font as the figures.
import sys
sys.path.insert(0, "scripts")
from table_style import render_table

COLUMNS = [
    {"key": "wbgene", "label": "WBGene ID", "width": 0.16, "wrap": True},
    {"key": "seqname", "label": "Sequence name", "width": 0.12, "wrap": True},
    {"key": "public_name", "label": "Gene", "width": 0.11, "wrap": True},
    {"key": "category", "label": "Functional category", "width": 0.26, "wrap": True},
    {"key": "has_symbol", "label": "Has gene symbol?", "width": 0.10, "align": "center",
     "wrap": True, "format": lambda v: "Yes" if v else "No"},
    {"key": "go_annotated", "label": "GO annotated?", "width": 0.09, "align": "center",
     "wrap": True, "format": lambda v: "Yes" if v else "No"},
    {"key": "rank_score", "label": "Rank (Score)", "width": 0.08, "align": "center", "wrap": True},
    {"key": "rank_var", "label": "Rank (Score/var)", "width": 0.08, "align": "center", "wrap": True},
]
n_pages = render_table(
    table_s2_sorted, COLUMNS,
    title="Table S2. Functional category for all 61 high-confidence ATFS-1 target genes.",
    filename="table_s2_regulon_annotation",
    footnote=("Sorted by functional category (Folding/QC first) then by Score/variability "
              "rank within category. “Has gene symbol” and “GO annotated” are "
              "related but distinct axes - 28 of 61 genes carry no public gene symbol, 19 of "
              "61 carry no GO annotation, and only 13 genes are both; see gate_decisions.md."),
)
print(f"Table S2 rendered to {n_pages} page(s).")

Wrote tables/table_s2_regulon_annotation.pdf (3 page(s)); preview PNG(s): ['tables/table_s2_regulon_annotation_page1.png', 'tables/table_s2_regulon_annotation_page2.png', 'tables/table_s2_regulon_annotation_page3.png']
Table S2 rendered to 3 page(s).
